# MICrONs Project 7 — From Structure to Function

**Dataset:** `minnie65_public` (mouse V1 + HVAs, ~1mm³)

**Goal:** Test whether synaptically connected neurons are more functionally correlated, and whether the two networks share topological organisation.

**Data already on disk:**
- `microns.h5` — 14 sessions of calcium imaging responses (neurons × 75 timepoints, 464 trials/session)
- Auth token saved at system level — `CAVEclient` works without extra setup

**Note on functional data:** Responses are stimulus-evoked (Monet2 natural-image stimuli), not resting-state. Pairwise correlations therefore reflect both shared tuning and direct synaptic coupling. This is acknowledged as a limitation: we cannot cleanly separate signal correlations from noise correlations without signal-subtracted noise correlation analysis (beyond scope here).

---
### Pipeline overview
1. Setup & matched-neuron go/no-go
2. Session selection — pick the session with the most matched neurons
3. Structural adjacency matrix (directed, E/I separated)
4. Functional correlation matrix (Pearson + partial)
5. Structure–function test (permutation)
6. Network metrics vs configuration-model null
7. Distance control

---
## 00 — Imports & constants

In [ ]:
import warnings, os, h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats as stats
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

import microns_datacleaner as mic
import microns_datacleaner.filters as fl
import microns_datacleaner.remapper as rem

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_style('whitegrid')
RNG = np.random.default_rng(42)

# Paths are relative to this notebook (Leo/), so step up one level
DATA_DIR  = '../data'
H5_PATH   = '../data/functional/microns_functional.h5'
VERSION   = 1718
GO_THRESHOLD = 200                 # minimum matched neurons to proceed

print('h5 file present:', os.path.exists(H5_PATH))

---
## 01 — Build unit table & go/no-go checkpoint

`process_nucleus_data(functional_data='best_only')` returns one row per neuron, keeping only the scan with the highest Digital-Twin `cc_abs` score when a neuron was recorded in multiple sessions. Columns relevant to us:
- `nucleus_id`, `pt_root_id` — structural identifiers
- `pt_position_x/y/z` — pial-surface coordinates in µm (already transformed by the package)
- `classification_system` — `'excitatory_neuron'` / `'inhibitory_neuron'` / `'nonneuron'`
- `strategy_axon`, `strategy_dendrite` — proofreading status
- `tuning_type` — `'matched'` if co-registered with functional data, else `'not_matched'`
- `session`, `scan_idx`, `functional_unit_id` — keys into `microns.h5`
- `cc_abs` — Digital-Twin cross-correlation score (data quality proxy)

In [ ]:
cleaner = mic.MicronsDataCleaner(datadir=DATA_DIR, version=VERSION, download_policy='minimum')
cleaner.download_nucleus_data()
# functional_data='all' is required to keep session, scan_idx and functional_unit_id columns.
# 'best_only' drops them by design (see microns_datacleaner docs).
units_all, segments = cleaner.process_nucleus_data(functional_data='all')

print(f'Total units (all rows; multi-scan neurons appear multiple times): {len(units_all):,}')
print(f'Columns: {list(units_all.columns)}')
units_all.head(3)

In [ ]:
# ── Matched subset: functionally co-registered neurons only ───────────────────
# 'matched' means the neuron has a functional_unit_id linking it to microns.h5
matched = fl.filter_neurons(units_all, tuning='matched').copy()
matched = matched.reset_index(drop=True)

print(f'Functionally matched neurons : {len(matched):,}')
print(f'Excitatory matched           : {(matched["classification_system"] == "excitatory_neuron").sum():,}')
print(f'Inhibitory matched           : {(matched["classification_system"] == "inhibitory_neuron").sum():,}')

if len(matched) >= GO_THRESHOLD:
    print(f'\n✓  GO — {len(matched):,} matched neurons exceeds threshold ({GO_THRESHOLD})')
else:
    print(f'\n✗  NO-GO — only {len(matched)} matched neurons (threshold {GO_THRESHOLD})')
    print('   Pivot to Project 12 (structural-only community detection).')

---
## 02 — Session selection

Each matched neuron belongs to exactly one session (`best_only` policy). We want the single session whose matched neurons have the largest overlap with the structural dataset, so that the matched subset used for all subsequent analysis is as large as possible.

**Why one session?** The functional correlation matrix must be a dense N×N matrix — there is no principled way to fill in cross-session pairwise correlations without making additional model assumptions. Using one session gives us a complete, internally consistent functional network.

In [ ]:
# ── Count matched neurons per session ─────────────────────────────────────────
# session column is stored as float (e.g. 7.0 for session 7); scan_idx similarly
# h5 keys are formatted as '{session}_{scan_idx}'  (e.g. '7_4')

with h5py.File(H5_PATH, 'r') as f:
    h5_sessions = set(f['sessions'].keys())

matched['h5_key'] = (
    matched['session'].astype(int).astype(str) + '_' +
    matched['scan_idx'].astype(int).astype(str)
)

session_counts = (
    matched[matched['h5_key'].isin(h5_sessions)]
    .groupby('h5_key')
    .size()
    .sort_values(ascending=False)
    .rename('n_matched_neurons')
)

print('Matched neurons per session (in h5):')
print(session_counts.to_string())

BEST_SESSION = session_counts.index[0]
print(f'\nSelected session: {BEST_SESSION}  ({session_counts.iloc[0]:,} matched neurons)')

In [ ]:
# ── Final analysis cohort: matched neurons in the chosen session ───────────────
# Filter to the chosen (session, scan_idx). A neuron may legitimately appear more
# than once in the same scan (different functional_unit_id mappings); keep only
# the row with the highest cc_abs to get one canonical scan per neuron.
cohort = (matched[matched['h5_key'] == BEST_SESSION]
          .sort_values('cc_abs', ascending=False)
          .drop_duplicates(subset='pt_root_id', keep='first')
          .reset_index(drop=True))

# Canonical integer index: neuron i ↔ cohort.iloc[i]
N = len(cohort)
id_to_idx = {int(rid): i for i, rid in enumerate(cohort['pt_root_id'])}

print(f'Analysis cohort size N = {N}')
print(f'Excitatory: {(cohort["classification_system"]=="excitatory_neuron").sum():,}')
print(f'Inhibitory: {(cohort["classification_system"]=="inhibitory_neuron").sum():,}')
cohort[['nucleus_id','pt_root_id','classification_system','cell_type',
        'layer','brain_area','pt_position_x','pt_position_y','pt_position_z',
        'functional_unit_id','h5_key']].head()

---
## 03 — Structural adjacency matrix

We download **all synapses where both pre and post are in our cohort**. The `size` column is the total synaptic contact area in nm² — we use this as edge weight (proxy for synaptic strength).

Convention: `W[i, j]` = total synaptic input from neuron `j` → neuron `i` (post × pre).

We build three matrices: all edges, excitatory-source only, inhibitory-source only. Collapsing E and I would obscure the sign of the influence.

In [ ]:
# ── Download synapses within cohort ───────────────────────────────────────────
cohort_ids = cohort['pt_root_id']
print(f'Downloading synapses among {N} cohort neurons ...')
cleaner.download_synapse_data(cohort_ids, cohort_ids)
cleaner.merge_synapses(syn_table_name='cohort_synapses')

In [ ]:
syn_path = os.path.join(DATA_DIR, str(VERSION), 'raw', 'cohort_synapses.csv')
synapses = pd.read_csv(syn_path)
print(f'Synapse rows: {len(synapses):,}')
print(f'Columns: {list(synapses.columns)}')
synapses.head(3)

In [ ]:
# ── Build sparse adjacency matrices ───────────────────────────────────────────
exc_ids = set(cohort.loc[cohort['classification_system']=='excitatory_neuron', 'pt_root_id'].astype(int))
inh_ids = set(cohort.loc[cohort['classification_system']=='inhibitory_neuron', 'pt_root_id'].astype(int))

def adjacency_from_synapses(syn: pd.DataFrame) -> sp.csr_matrix:
    """Build N×N sparse matrix; W[post, pre] = sum of synaptic sizes."""
    pre  = syn['pre_pt_root_id'].astype(int).map(id_to_idx)
    post = syn['post_pt_root_id'].astype(int).map(id_to_idx)
    mask = pre.notna() & post.notna()
    pre, post = pre[mask].astype(int).values, post[mask].astype(int).values
    w = syn.loc[mask, 'size'].values.astype(float)
    agg = pd.DataFrame({'post': post, 'pre': pre, 'w': w})
    agg = agg.groupby(['post', 'pre'], as_index=False)['w'].sum()
    return sp.csr_matrix((agg['w'].values, (agg['post'].values, agg['pre'].values)),
                         shape=(N, N))

W_all = adjacency_from_synapses(synapses)
W_exc = adjacency_from_synapses(synapses[synapses['pre_pt_root_id'].isin(exc_ids)])
W_inh = adjacency_from_synapses(synapses[synapses['pre_pt_root_id'].isin(inh_ids)])

print(f'W_all  shape={W_all.shape}  nnz={W_all.nnz:,}  density={W_all.nnz/N**2:.4%}')
print(f'W_exc  nnz={W_exc.nnz:,}  density={W_exc.nnz/N**2:.4%}')
print(f'W_inh  nnz={W_inh.nnz:,}  density={W_inh.nnz/N**2:.4%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, mat, title in zip(axes, [W_all, W_exc, W_inh], ['All', 'Excitatory src', 'Inhibitory src']):
    deg = np.array(mat.sum(axis=1)).ravel()  # weighted in-degree
    ax.hist(deg[deg > 0], bins=50, color='steelblue', alpha=0.8)
    ax.set_title(f'{title} — weighted in-degree')
    ax.set_xlabel('Total synaptic input (nm²)')
plt.tight_layout()
plt.savefig('fig_structural_degrees.png', dpi=150)
plt.show()

---
## 04 — Functional correlation matrix

**Data:** `microns.h5`, session `BEST_SESSION`. Each of 464 trials contains a `responses` array `(N_session, 75)` — the deconvolved calcium signal at 75 stimulus time points.

**Strategy:** concatenate all 464 trials → matrix `R` of shape `(N_cohort, 464×75)`. This gives T ≈ 34,800 time points >> N, so the covariance matrix is full-rank and we can invert it directly for partial correlations (no regularisation needed).

**Two outputs:**
- `C_pearson` — raw Pearson. Inflated by shared stimulus drive and global brain state. Reported for transparency.
- `C_partial` — partial correlation via precision matrix. Controls for all shared linear dependencies. Used as **primary metric** in all subsequent tests.

In [ ]:
# ── Load and align response data ───────────────────────────────────────────────
with h5py.File(H5_PATH, 'r') as f:
    sess = f[f'sessions/{BEST_SESSION}']
    h5_unit_ids = sess['meta/unit_ids'][:].astype(int)   # functional_unit_id in h5
    trial_keys  = sorted(sess['trials'].keys(), key=int)
    # Each trial array has shape (N_session, T_k): rows = units, cols = timepoints
    trial_list  = [sess[f'trials/{k}/responses'][:] for k in trial_keys]

# Map functional_unit_id → row index in the responses array
h5_uid_to_row = {uid: row for row, uid in enumerate(h5_unit_ids)}
cohort_func_ids = cohort['functional_unit_id'].astype(int).values

matched_mask = np.array([uid in h5_uid_to_row for uid in cohort_func_ids])
row_order    = [h5_uid_to_row[uid] for uid in cohort_func_ids if uid in h5_uid_to_row]

if not matched_mask.all():
    n_missing = (~matched_mask).sum()
    print(f'WARNING: {n_missing} cohort neurons not found in h5 — dropping them')
    cohort   = cohort[matched_mask].reset_index(drop=True)
    N        = len(cohort)
    id_to_idx = {int(rid): i for i, rid in enumerate(cohort['pt_root_id'])}
    row_order = [h5_uid_to_row[int(uid)] for uid in cohort['functional_unit_id']]
    # Rebuild adjacency matrices against the shrunk id_to_idx
    W_all = adjacency_from_synapses(synapses)
    W_exc = adjacency_from_synapses(synapses[synapses['pre_pt_root_id'].isin(exc_ids)])
    W_inh = adjacency_from_synapses(synapses[synapses['pre_pt_root_id'].isin(inh_ids)])

# Select cohort rows from each trial → each block is (N_cohort, T_k)
R_list  = [trial[row_order, :] for trial in trial_list]
R       = np.concatenate(R_list, axis=1)         # (N_cohort, T_total)
T_total = R.shape[1]

print(f'Response matrix R: {N} neurons × {T_total} time points  (T >> N: {T_total > N})')

In [ ]:
# ── Pearson correlation ────────────────────────────────────────────────────────
C_pearson = np.corrcoef(R)                  # (N, N)
np.fill_diagonal(C_pearson, 0.0)
print(f'C_pearson  mean={C_pearson[C_pearson!=0].mean():.4f}  '
      f'std={C_pearson[C_pearson!=0].std():.4f}')

In [ ]:
# ── Partial correlation via precision matrix ───────────────────────────────────
# T >> N → covariance matrix is full-rank; direct inversion is valid.
# Partial corr: PC_ij = -K_ij / sqrt(K_ii * K_jj)  where K = Cov^{-1}
cov  = np.cov(R)                             # (N, N)
prec = np.linalg.inv(cov)                    # precision matrix
D_inv = 1.0 / np.sqrt(np.diag(prec))
C_partial = -prec * np.outer(D_inv, D_inv)
np.fill_diagonal(C_partial, 0.0)
print(f'C_partial  mean={C_partial[C_partial!=0].mean():.4f}  '
      f'std={C_partial[C_partial!=0].std():.4f}')

In [ ]:
PLOT_N = min(200, N)
idx_sub = np.sort(RNG.choice(N, PLOT_N, replace=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mat, title in zip(axes, [C_pearson, C_partial], ['Pearson', 'Partial']):
    sub = mat[np.ix_(idx_sub, idx_sub)]
    vmax = np.percentile(np.abs(sub), 99)
    im = ax.imshow(sub, cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='none')
    ax.set_title(f'{title} correlation  (n={PLOT_N})')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('fig_correlation_matrices.png', dpi=150)
plt.show()

---
## 05 — Structure–function test

### Test A: Are connected pairs more correlated? (Mann-Whitney U)
**Why permutation, not t-test:** pairwise correlation values share neurons, so they are not independent. Permuting connectivity labels destroys the structure-function correspondence while preserving the marginal distributions of both connectivity and correlation — giving a valid null.

### Test B: Does synapse weight predict correlation magnitude? (Spearman ρ)
Only on connected pairs to avoid the structural zero-inflation dominating the rank.

### Direction note
`W_all` is directed. For the binary connectivity label we use: pair `(i,j)` is *connected* if `W[i,j] > 0` OR `W[j,i] > 0`. Directional breakdown (E→E, E→I, I→E, I→I) reported separately as an extension.

In [ ]:
# ── Extract upper-triangle pairs ───────────────────────────────────────────────
i_idx, j_idx = np.triu_indices(N, k=1)
n_pairs = len(i_idx)

W_dense = W_all.toarray()
# Binary symmetric: connected if any synapse in either direction
connected = ((W_dense + W_dense.T) > 0).astype(np.int8)
conn_label  = connected[i_idx, j_idx]        # 1=connected, 0=not
syn_weight  = (W_dense[i_idx, j_idx] +
               W_dense[j_idx, i_idx])         # total synaptic area, both directions
fc_pearson  = C_pearson[i_idx, j_idx]
fc_partial  = C_partial[i_idx, j_idx]

n_conn = int(conn_label.sum())
n_disc = n_pairs - n_conn
print(f'Total pairs   : {n_pairs:,}')
print(f'Connected     : {n_conn:,}  ({n_conn/n_pairs:.3%})')
print(f'Unconnected   : {n_disc:,}')

In [ ]:
# ── Test A: permutation Mann-Whitney ──────────────────────────────────────────
N_PERM = 10_000

def perm_mannwhitney(vals, labels, n_perm, rng):
    obs, _ = stats.mannwhitneyu(vals[labels==1], vals[labels==0], alternative='greater')
    null = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng.permutation(labels)
        null[k], _ = stats.mannwhitneyu(vals[perm==1], vals[perm==0], alternative='greater')
    return obs, (null >= obs).mean(), null

print('Test A (partial) ...')
obs_pc, p_pc, null_pc = perm_mannwhitney(fc_partial, conn_label, N_PERM, RNG)
print(f'  Partial corr : U={obs_pc:.0f}  perm-p={p_pc:.4f}')

print('Test A (Pearson) ...')
obs_pe, p_pe, null_pe = perm_mannwhitney(fc_pearson, conn_label, N_PERM, RNG)
print(f'  Pearson corr : U={obs_pe:.0f}  perm-p={p_pe:.4f}')
print('  (Pearson reported for comparison; inflation risk noted in Methods)')

In [ ]:
# ── Test B: Spearman ρ (synapse weight → partial correlation) ─────────────────
# Restricted to connected pairs to avoid zero-inflation of synapse counts
mask_conn = conn_label == 1
rho_pc, _ = stats.spearmanr(syn_weight[mask_conn], fc_partial[mask_conn])
rho_pe, _ = stats.spearmanr(syn_weight[mask_conn], fc_pearson[mask_conn])
print(f'Test B — synapse weight vs partial corr  : Spearman ρ = {rho_pc:.4f}')
print(f'         synapse weight vs Pearson corr  : Spearman ρ = {rho_pe:.4f}')
print('(Permutation p for ρ: permute the weight vector among connected pairs)')

In [ ]:
# ── Permutation p for Spearman ρ ──────────────────────────────────────────────
w_conn  = syn_weight[mask_conn]
fc_conn = fc_partial[mask_conn]
null_rho = np.array([stats.spearmanr(RNG.permutation(w_conn), fc_conn)[0]
                     for _ in range(N_PERM)])
p_rho = (null_rho >= rho_pc).mean()
print(f'Permutation p for Spearman ρ: {p_rho:.4f}')

In [ ]:
# ── E/I breakdown (extension) ─────────────────────────────────────────────────
W_exc_d = W_exc.toarray()
W_inh_d = W_inh.toarray()

for label, W_sub in [('E→* (exc pre)', W_exc_d), ('I→* (inh pre)', W_inh_d)]:
    sub_conn = ((W_sub + W_sub.T) > 0).astype(np.int8)[i_idx, j_idx]
    n_sub = int(sub_conn.sum())
    if n_sub < 10:
        print(f'{label}: only {n_sub} pairs — skipping')
        continue
    conn_fc = fc_partial[sub_conn == 1]
    disc_fc = fc_partial[sub_conn == 0]
    u, _ = stats.mannwhitneyu(conn_fc, disc_fc, alternative='greater')
    print(f'{label}: {n_sub} connected pairs,  mean Δpartial = {conn_fc.mean()-disc_fc.mean():.4f}  U={u:.0f}')

In [ ]:
# ── Figure ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
df_plot = pd.DataFrame({'partial_corr': fc_partial, 'connected': conn_label.astype(bool)})
sns.violinplot(data=df_plot, x='connected', y='partial_corr', ax=ax,
               palette={False:'steelblue', True:'salmon'}, inner='quartile', cut=0)
ax.set_xticklabels(['Unconnected', 'Connected'])
ax.set_title(f'Partial corr by connectivity\nperm p = {p_pc:.4f}')
ax.set_ylabel('Partial correlation')

ax = axes[1]
ax.hist(null_pc, bins=60, color='grey', alpha=0.7, density=True)
ax.axvline(obs_pc, color='red', lw=2, label=f'Observed U={obs_pc:.0f}')
ax.set_title('Permutation null (Test A, partial)')
ax.set_xlabel('Mann-Whitney U')
ax.legend()

ax = axes[2]
ax.hexbin(syn_weight[mask_conn], fc_partial[mask_conn], gridsize=40,
          cmap='viridis', mincnt=1)
ax.set_xlabel('Synapse weight (nm²)')
ax.set_ylabel('Partial correlation')
ax.set_title(f'Test B: synapse weight vs partial corr\nSpearman ρ={rho_pc:.4f}  perm p={p_rho:.4f}')

plt.tight_layout()
plt.savefig('fig_sf_tests.png', dpi=150)
plt.show()

---
## 06 — Network metrics vs configuration-model null

Metrics: degree distribution, clustering coefficient C, average path length L, small-world coefficient σ = (C/C_rand)/(L/L_rand).

Null model: configuration model — preserves degree sequence, randomises wiring. Multi-edges and self-loops removed.

**Functional graph thresholding note:** to convert the weighted functional matrix to a binary graph we threshold at the 95th percentile of |partial correlation|. This choice affects all graph metrics. We report sensitivity to this threshold at the end of the cell.

In [ ]:
# ── Build graphs ───────────────────────────────────────────────────────────────
G_struct = nx.from_scipy_sparse_array(W_all, create_using=nx.DiGraph)

FUNC_PCT = 95
func_thr = np.percentile(np.abs(C_partial[C_partial != 0]), FUNC_PCT)
C_bin = (np.abs(C_partial) > func_thr).astype(float)
np.fill_diagonal(C_bin, 0)
G_func = nx.from_numpy_array(C_bin)  # undirected

print(f'G_struct: {G_struct.number_of_nodes()} nodes, {G_struct.number_of_edges()} edges')
print(f'G_func  : {G_func.number_of_nodes()} nodes, {G_func.number_of_edges()} edges  (threshold={func_thr:.4f})')

In [ ]:
def lcc(G):
    if G.is_directed():
        return G.subgraph(max(nx.weakly_connected_components(G), key=len)).copy()
    return G.subgraph(max(nx.connected_components(G), key=len)).copy()

def graph_metrics(G, path_sample=500, rng=None):
    rng = rng or np.random.default_rng()
    G_cc = lcc(G)
    G_u  = G_cc.to_undirected() if G.is_directed() else G_cc
    C    = nx.average_clustering(G_u)
    nodes = list(G_u.nodes())
    sample = rng.choice(nodes, min(path_sample, len(nodes)), replace=False)
    L = np.mean([l for s in sample
                 for l in nx.single_source_shortest_path_length(G_u, s).values() if l > 0])
    return {'n_lcc': G_cc.number_of_nodes(), 'C': C, 'L': L}


m_s = graph_metrics(G_struct, rng=RNG)
m_f = graph_metrics(G_func,   rng=RNG)
print('Structural:', m_s)
print('Functional:', m_f)

In [ ]:
# ── Configuration-model null ───────────────────────────────────────────────────
# Increase N_NULL to 1000 for final results (100 is fast for development)
N_NULL = 100

def null_distribution(G, n_null, rng, label=''):
    C_null, L_null = [], []
    in_seq  = [d for _, d in G.in_degree()]  if G.is_directed() else None
    out_seq = [d for _, d in G.out_degree()] if G.is_directed() else None
    deg_seq = [d for _, d in G.degree()]     if not G.is_directed() else None

    for k in range(n_null):
        if G.is_directed():
            Gn = nx.DiGraph(nx.directed_configuration_model(in_seq, out_seq, seed=rng))
        else:
            Gn = nx.Graph(nx.configuration_model(deg_seq, seed=rng))
        Gn.remove_edges_from(nx.selfloop_edges(Gn))
        m = graph_metrics(Gn, path_sample=200, rng=rng)
        C_null.append(m['C']); L_null.append(m['L'])
        if (k+1) % 25 == 0: print(f'  {label} {k+1}/{n_null}')
    return np.array(C_null), np.array(L_null)

print('Structural null ...')
nc_s, nl_s = null_distribution(G_struct, N_NULL, RNG, 'struct')
print('Functional null ...')
nc_f, nl_f = null_distribution(G_func,   N_NULL, RNG, 'func')

In [ ]:
# ── Small-world coefficient σ ─────────────────────────────────────────────────
def sigma(C_real, L_real, C_null, L_null):
    Cr, Lr = C_null.mean(), L_null.mean()
    s = (C_real / Cr) / (L_real / Lr)
    return s, Cr, Lr

for name, m, nc, nl in [('Structural', m_s, nc_s, nl_s), ('Functional', m_f, nc_f, nl_f)]:
    s, Cr, Lr = sigma(m['C'], m['L'], nc, nl)
    zC = (m['C'] - nc.mean()) / nc.std()
    zL = (m['L'] - nl.mean()) / nl.std()
    print(f'{name}:')
    print(f'  C={m["C"]:.4f}  C_rand={Cr:.4f}  z={zC:+.2f}')
    print(f'  L={m["L"]:.4f}  L_rand={Lr:.4f}  z={zL:+.2f}')
    print(f'  σ = {s:.3f}  →  {"SMALL-WORLD (σ>1)" if s > 1 else "not small-world"}')

In [ ]:
# ── Degree distribution — MLE fitting ─────────────────────────────────────────
# Visual log-log is not a valid power-law test (Clauset et al. 2009).
# Use the `powerlaw` package (MLE + likelihood-ratio tests against alternatives).
try:
    import powerlaw
    for name, deg_arr in [
        ('struct in-degree',  np.array([d for _,d in G_struct.in_degree()],  dtype=float)),
        ('struct out-degree', np.array([d for _,d in G_struct.out_degree()], dtype=float)),
        ('func degree',       np.array([d for _,d in G_func.degree()],       dtype=float)),
    ]:
        d = deg_arr[deg_arr > 0]
        fit = powerlaw.Fit(d, discrete=True, verbose=False)
        R_exp, p_exp = fit.distribution_compare('power_law', 'exponential')
        R_log, p_log = fit.distribution_compare('power_law', 'lognormal')
        print(f'{name:25s}  α={fit.power_law.alpha:.3f}  '
              f'vs-exp R={R_exp:.2f}(p={p_exp:.3f})  '
              f'vs-lognormal R={R_log:.2f}(p={p_log:.3f})')
except ImportError:
    print('Install powerlaw for MLE fitting: pip install powerlaw')

---
## 07 — Distance control

**Why this matters:** nearby neurons connect more often and correlate more regardless of direct connectivity. Any apparent structure-function correspondence could be a spatial proximity artefact.

**Two complementary approaches:**
1. **Distance-binned test** — repeat Test A (Mann-Whitney) within each distance decile. If the effect survives at all distances, it is not purely a proximity artefact.
2. **Partial Spearman ρ** — rank-transform both synapse weight and partial correlation, regress out distance ranks, correlate residuals. Isolates structure-function correspondence net of distance.

Positions from `pt_position_x/y/z` are in µm (pial-surface coordinates, already transformed by `microns_datacleaner`).

In [ ]:
from scipy.spatial.distance import pdist, squareform

coords = cohort[['pt_position_x', 'pt_position_y', 'pt_position_z']].values   # µm
D_sq   = squareform(pdist(coords))                  # N × N, µm
dist_pairs = D_sq[i_idx, j_idx]

print(f'Pairwise distance: min={dist_pairs.min():.1f} µm  '
      f'median={np.median(dist_pairs):.1f} µm  max={dist_pairs.max():.1f} µm')

In [ ]:
# ── Distance-binned Mann-Whitney ───────────────────────────────────────────────
bin_edges = np.percentile(dist_pairs, np.linspace(0, 100, 11))   # deciles
bin_results = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask_bin = (dist_pairs >= lo) & (dist_pairs < hi)
    lbl = conn_label[mask_bin]
    fc  = fc_partial[mask_bin]
    nc, nd = lbl.sum(), (lbl==0).sum()
    if nc < 5 or nd < 5:
        bin_results.append({'dist_mid': (lo+hi)/2, 'delta': np.nan, 'p_perm': np.nan, 'n_conn': nc})
        continue
    delta = fc[lbl==1].mean() - fc[lbl==0].mean()
    null_d = np.array([RNG.permutation(fc)[RNG.permutation(lbl)==1].mean() -
                       RNG.permutation(fc)[RNG.permutation(lbl)==0].mean()
                       for _ in range(1_000)])
    p = (null_d >= delta).mean()
    bin_results.append({'dist_mid': (lo+hi)/2, 'delta': delta, 'p_perm': p, 'n_conn': nc})

bin_df = pd.DataFrame(bin_results)
print(bin_df.to_string(float_format='{:.4f}'.format))

In [ ]:
# ── Partial Spearman ρ (controlling for distance) ─────────────────────────────
from scipy.stats import rankdata

def partial_spearman(x, y, z):
    """Spearman partial corr of x,y controlling for z.
    Ranks all variables, regresses z-ranks out of x-ranks and y-ranks, 
    returns Pearson r on the residuals."""
    rx, ry, rz = map(lambda a: rankdata(a).astype(float), (x, y, z))
    rz_c = rz - rz.mean()
    def resid(a):
        return a - (np.dot(rz_c, a) / np.dot(rz_c, rz_c)) * rz_c
    r, p = stats.pearsonr(resid(rx), resid(ry))
    return r, p

m_conn = mask_conn
rho_ps, p_ps = partial_spearman(
    fc_partial[m_conn], syn_weight[m_conn], dist_pairs[m_conn]
)
print(f'Partial Spearman ρ (partial corr ~ synapse weight | distance) = {rho_ps:.4f}')
print(f'  Parametric p = {p_ps:.4f}  (for screening; run permutation for final report)')
print(f'  Interpretation: positive → synapse weight predicts functional correlation '
       f'beyond what distance alone explains.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hexbin(dist_pairs, fc_partial, gridsize=50, cmap='Blues', mincnt=1)
ax.set_xlabel('Distance (µm)'); ax.set_ylabel('Partial correlation')
ax.set_title('Distance vs partial corr (all pairs)')

ax = axes[1]
bin_edges_20 = np.percentile(dist_pairs, np.linspace(0, 100, 21))
frac = [conn_label[(dist_pairs>=lo)&(dist_pairs<hi)].mean()
        for lo, hi in zip(bin_edges_20[:-1], bin_edges_20[1:])]
mids = [(lo+hi)/2 for lo, hi in zip(bin_edges_20[:-1], bin_edges_20[1:])]
ax.plot(mids, frac, 'o-', color='tomato')
ax.set_xlabel('Distance (µm)'); ax.set_ylabel('P(connected)')
ax.set_title('Connection probability vs distance')

ax = axes[2]
valid = bin_df.dropna()
cols = ['red' if p < 0.05 else 'grey' for p in valid['p_perm']]
ax.bar(range(len(valid)), valid['delta'], color=cols)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(range(len(valid)))
ax.set_xticklabels([f'{d:.0f}µm' for d in valid['dist_mid']], rotation=45, ha='right')
ax.set_ylabel('Δ partial corr (conn−unconn)')
ax.set_title('Distance-binned effect (red = perm p<0.05)')

plt.tight_layout()
plt.savefig('fig_distance_control.png', dpi=150)
plt.show()

---
## 08 — Summary table

In [ ]:
summary = {
    'N_cohort'                      : N,
    'session_used'                  : BEST_SESSION,
    'T_total_timepoints'            : T_total,
    'N_structural_edges'            : W_all.nnz,
    'N_functional_edges_thresh'     : G_func.number_of_edges(),
    'func_threshold_pct'            : FUNC_PCT,
    'TestA_U_partial'               : obs_pc,
    'TestA_perm_p_partial'          : p_pc,
    'TestA_U_Pearson'               : obs_pe,
    'TestA_perm_p_Pearson'          : p_pe,
    'TestB_Spearman_rho_partial'    : rho_pc,
    'TestB_perm_p'                  : p_rho,
    'struct_C'                      : m_s['C'],
    'struct_L'                      : m_s['L'],
    'struct_sigma'                  : sigma(m_s['C'], m_s['L'], nc_s, nl_s)[0],
    'func_C'                        : m_f['C'],
    'func_L'                        : m_f['L'],
    'func_sigma'                    : sigma(m_f['C'], m_f['L'], nc_f, nl_f)[0],
    'partial_Spearman_dist_controlled': rho_ps,
}

summary_df = pd.Series(summary).to_frame('value')
summary_df.index.name = 'metric'
print(summary_df.to_string(float_format='{:.4f}'.format))
summary_df.to_csv('results_summary.csv')
print('\nSaved results_summary.csv')